In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Remember a store manager's preferences with state and Memory Bank

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Two kinds of memory

An agent that helps the same person every day should not ask the same questions every day. ADK gives you two places to keep what it learns:

- **User state.** [Session state](https://google.github.io/adk-docs/sessions/state/) keys that start with `user:` belong to the user, not to one conversation. Every new session for that user starts with them, and the agent's instruction can read them directly, for example `{user:huddle_time?}`.
- **Memory Bank.** [Vertex AI Memory Bank](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/overview) reads past conversations, extracts the facts worth keeping, and stores them per user. In a later conversation the agent searches those memories.

Use user state for settings you know the exact shape of. Use Memory Bank for facts that come up in conversation and that you did not plan fields for.

### The huddle assistant

A store manager runs a short team huddle at the start of the day. This agent prepares it. It saves the manager's huddle time and the order they like the topics in as user state, and it reads what Memory Bank remembers about the manager before every turn.

<img width="60%" src="../../docs/diagrams/q06.png" alt="A huddle assistant that keeps preferences in user state and past conversations in Memory Bank" />

### Objectives

In this tutorial, you will learn how an agent remembers a user across conversations.

You will complete the following tasks:

- Create an Agent Engine instance to use its Memory Bank
- Build an agent that saves preferences as user state and reads memories before every turn
- Send a conversation to Memory Bank and read the memories it generated
- Start a new conversation and see the agent recall both

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery
- Vertex AI Agent Engine

Learn about [Gemini on Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing), [BigQuery pricing](https://cloud.google.com/bigquery/pricing), [Vertex AI Agent Engine pricing](https://cloud.google.com/vertex-ai/pricing#vertex-ai-agent-engine), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded during setup, in your own namespace. Set your project ID and the namespace you chose.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# This notebook sits two folders below the repository root, where the shared store tools live
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, SDKs announce renamed classes with a FutureWarning, and the Gen AI SDK logs a note
# whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

import re

import agentplatform
from google.adk.agents import LlmAgent
from google.adk.memory import VertexAiMemoryBankService
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext, preload_memory
from google.genai import types

from agents.cymbal_store_ops.tools.domain_tools import (
    get_osa_exceptions,
    get_shrink_signals,
    get_traffic_and_backlog,
)

### Choose the model

The agent in this tutorial uses Gemini 3.8 Flash. The retry options make the SDK retry a request that fails with a temporary error, such as a 429 or a 500, instead of failing the turn.

In [4]:
model = Gemini(
    model="gemini-3.8-flash",
    retry_options=types.HttpRetryOptions(attempts=4, initial_delay=2.0),
)

## Create an Agent Engine instance for Memory Bank

Memory Bank is part of Vertex AI Agent Engine. To use it, you need an Agent Engine instance. It does not need a deployed agent: an empty instance is enough. In the run shown, creating one took a few seconds; it can take up to a couple of minutes.

In [5]:
LOCATION = "us-central1"

client = agentplatform.Client(project=PROJECT_ID, location=LOCATION)
agent_engine = client.agent_engines.create()

AGENT_ENGINE_NAME = agent_engine.api_resource.name
AGENT_ENGINE_ID = AGENT_ENGINE_NAME.split("/")[-1]
print(f"Created Agent Engine: {AGENT_ENGINE_NAME}")

Created Agent Engine: projects/763419985448/locations/us-central1/reasoningEngines/1237728485724651520


## Build the agent

### Save preferences as user state

`remember_preferences` writes two `user:` keys: the huddle time, and the topics to cover in order. It checks both values before saving them.

In [6]:
FOCUS_AREAS = ("bopis", "on_shelf", "shrink")


def remember_preferences(
    tool_context: ToolContext, huddle_time: str = "", focus_areas: list[str] | None = None
) -> dict:
    """Save the manager's huddle preferences for every future session.

    Args:
        huddle_time: 24-hour HH:MM, e.g. "08:45". Empty keeps the remembered one.
        focus_areas: what to cover first, in order, from: bopis, on_shelf, shrink.
    """
    if huddle_time and not re.fullmatch(r"([01]\d|2[0-3]):[0-5]\d", huddle_time):
        return {"status": "ERROR", "error_details": f"huddle_time {huddle_time!r} must be HH:MM, like 08:45"}
    unknown = [area for area in focus_areas or [] if area not in FOCUS_AREAS]
    if unknown:
        return {"status": "ERROR", "error_details": f"unknown focus areas {unknown}; choose from {FOCUS_AREAS}"}
    if huddle_time:
        tool_context.state["user:huddle_time"] = huddle_time
    if focus_areas:
        tool_context.state["user:focus_areas"] = ", ".join(focus_areas)
    return {
        "status": "SUCCESS",
        "huddle_time": tool_context.state.get("user:huddle_time", ""),
        "focus_areas": tool_context.state.get("user:focus_areas", ""),
    }

### Define the agent

The instruction reads the remembered preferences from state with `{user:huddle_time?}` and `{user:focus_areas?}`; the `?` means an empty value is fine. `preload_memory` searches Memory Bank with the user's message before every turn and adds the memories it finds to the request. The three store tools read the huddle facts for the signed-in store.

In [7]:
instruction = """You prepare a Cymbal Beauty store manager's start-of-day huddle and remember how they like it.
Remembered huddle time: {user:huddle_time?}. Remembered focus areas, in order: {user:focus_areas?}.
- When the manager states a huddle time or what to cover first, call remember_preferences.
- To prepare the huddle, cover the focus areas in the remembered order: bopis with get_traffic_and_backlog,
  on_shelf with get_osa_exceptions (limit 3), shrink with get_shrink_signals.
- Answer in at most four sentences, with the ids and counts from the tools and times in the store's
  local time with AM/PM. Never invent a number."""

agent = LlmAgent(
    name="memory_agent",
    model=model,
    description="Prepares a store manager's huddle and remembers their preferences across sessions.",
    instruction=instruction,
    tools=[
        preload_memory,
        remember_preferences,
        get_traffic_and_backlog,
        get_osa_exceptions,
        get_shrink_signals,
    ],
)

### Configure the runner

The runner combines the agent with a session service and a memory service. Sessions stay in memory in this notebook; memories go to the Agent Engine instance you created.

In [8]:
session_service = InMemorySessionService()
memory_service = VertexAiMemoryBankService(
    project=PROJECT_ID, location=LOCATION, agent_engine_id=AGENT_ENGINE_ID
)

runner = Runner(
    agent=agent,
    app_name="memory_agent",
    session_service=session_service,
    memory_service=memory_service,
)

USER_ID = "dana"
MANAGER = {"user:user_id": "U-M014", "user:store_id": "S-014", "user:role": "store_manager"}

Define a helper that sends one message in a given session and prints the tool calls and the answer.

In [9]:
async def ask(session, question: str) -> None:
    """Send one message in a session and print its tool calls and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(user_id=USER_ID, session_id=session.id, new_message=message):
        for call in event.get_function_calls():
            print(f"[{event.author}] calls {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}\n")

## First conversation: tell the agent your preferences

Dana says when her huddle is and what she always covers first, and mentions something about her team that has no field in the agent. This turn takes about 20 seconds:

In [10]:
session_1 = await session_service.create_session(app_name="memory_agent", user_id=USER_ID, state=MANAGER)

await ask(
    session_1,
    "My huddle is at 8:45 and I always want the BOPIS backlog first, then shelf gaps. "
    "Also, Jordan is training as a key holder this month. What should I tell the team this morning?",
)

[memory_agent] calls remember_preferences({'huddle_time': '08:45', 'focus_areas': ['bopis', 'on_shelf']})
[memory_agent] calls get_traffic_and_backlog({'hours': 4})
[memory_agent] calls get_osa_exceptions({'limit': 3})



We have 9 pending BOPIS orders promised between 9:30 AM and 11:00 AM to prioritize this morning. Next, complete backroom checks for three empty shelf products: P-0101 with 7 units in the backroom, P-0548 with 3 units, and P-0491 with 7 units. Finally, give a shout-out to Jordan as they begin training as a key holder this month.



The agent saved the huddle time and the two focus areas Dana named (`bopis`, `on_shelf`), then called the tool for each area in that order. Shrink was not named, so it was neither saved nor covered. The data: 9 pending pick-up orders promised between 9:30 AM and 11:00 AM, and three products with nothing on the shelf, P-0101 (7 in the backroom), P-0548 (3) and P-0491 (7). The wording varies from run to run.

The preferences are now user state:

In [11]:
session_1 = await session_service.get_session(app_name="memory_agent", user_id=USER_ID, session_id=session_1.id)
{key: value for key, value in session_1.state.items() if key.startswith("user:")}

{'user:user_id': 'U-M014',
 'user:store_id': 'S-014',
 'user:role': 'store_manager',
 'user:huddle_time': '08:45',
 'user:focus_areas': 'bopis, on_shelf'}

`user:huddle_time` and `user:focus_areas` were written by `remember_preferences`. The other three keys came from the state the session started with.

## Send the conversation to Memory Bank

Memory Bank reads a conversation and extracts the facts worth keeping, such as the note about Jordan. `add_events_to_memory` sends the session's events. With `wait_for_completion`, the call returns once the memories exist, which keeps this notebook in order.

In an application you send each conversation when it ends, from a callback or a background job, without waiting. `add_session_to_memory` does that, and Memory Bank generates the memories when the conversation goes idle.

In [12]:
await memory_service.add_events_to_memory(
    app_name="memory_agent",
    user_id=USER_ID,
    events=session_1.events,
    custom_metadata={"wait_for_completion": True},
)

Retrieve the memories stored for Dana. Memories are scoped by app name and user ID, so another user never sees them.

In [13]:
scope = {"app_name": "memory_agent", "user_id": USER_ID}

for retrieved in client.agent_engines.memories.retrieve(name=AGENT_ENGINE_NAME, scope=scope):
    print(f"- {retrieved.memory.fact}")

- My daily work huddle is at 8:45 AM, and I prefer to address the BOPIS (Buy Online, Pick Up In Store) backlog first, followed by shelf gaps.
- My team member, Jordan, is training as a key holder this month.


Memory Bank kept two facts: the huddle preferences and the note about Jordan. It writes them in the first person, and the wording varies from run to run.

## Second conversation: the agent remembers

Start a new session for the same user. The new session starts with the `user:` state, so the instruction already has the huddle time and the order. `preload_memory` adds the memories about Jordan to the request.

In [14]:
session_2 = await session_service.create_session(app_name="memory_agent", user_id=USER_ID, state=MANAGER)

await ask(session_2, "Prepare my huddle. Anything I should remember about the team?")

[memory_agent] calls get_traffic_and_backlog({})
[memory_agent] calls get_osa_exceptions({'limit': 3})



For your 8:45 AM huddle, you have 9 pending BOPIS orders promised between 9:30 AM and 11:00 AM. For on-shelf availability, prioritize backroom checks for P-0101 with 7 units in the backroom, P-0548 with 3 units in the backroom, and P-0491 with 7 units in the backroom, all of which currently have 0 on the shelf. Regarding your team, remember that Jordan is training as a key holder this month.



The agent used the 8:45 AM huddle time and covered pick-up orders, then shelf gaps, without being told again. The note about Jordan came from Memory Bank. No tool call shows for it because `preload_memory` runs before the model, not as a call the model makes. The counts match the first conversation.

Try these example phrases in a new session:

```
Move shrink to the front of my huddle from now on
What did I tell you about the team last time?
```

## Run the agent in the ADK developer UI

`agent.py` in this folder defines the same agent, plus `identify_demo_user` for signing in by ID, `load_memory` for searching memory on demand, and an `after_agent_callback` that sends each conversation to the memory service when a turn ends. The developer UI uses an in-memory memory service unless you point it at Memory Bank. From the repository root:

```bash
uv run python scripts/quickstart_apps.py 06-memory-agent
uv run adk web build/quickstart_apps --port 8001 --memory_service_uri=agentengine://<agent engine id>
```

On Agent Runtime, a deployed agent uses its own engine's Memory Bank without that flag.

## Cleaning up

Delete the Agent Engine instance and its memories to avoid charges.

In [15]:
delete_agent_engine = True

if delete_agent_engine:
    client.agent_engines.delete(name=AGENT_ENGINE_NAME, force=True)

## What's next

- [Session state](https://google.github.io/adk-docs/sessions/state/)
- [Memory in ADK](https://google.github.io/adk-docs/sessions/memory/)
- [Vertex AI Memory Bank](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/overview)
- [Quickstart 07: extract fields from documents](../07-document-extraction-agent/walkthrough.ipynb)